# Weeks 5–7: Complete Preprocessing and Classification Lab



## 1. Dependencies and imports
If imports fail, uncomment and run the installation line, then restart the kernel if requested. The remaining code uses installed NumPy, pandas and scikit-learn. Timings vary by hardware and thread settings.

In [13]:
# %pip install numpy pandas "scikit-learn>=1.2"
from pathlib import Path
from time import perf_counter
import json
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
import sklearn
SEED = 42



## 2. Get the dataset


In [14]:
dataset="https://raw.githubusercontent.com/swapnilsaurav/AIML/refs/heads/main/DPP/feature_engineering_construction_extraction/customer_churn.csv"
df = pd.read_csv(dataset)
print("Dataset shape:", df.shape)
print(df.head().to_string(index=False))


Dataset shape: (10000, 40)
customer_id  age  monthly_income  monthly_debt  tenure_months  orders_90d  returns_90d  total_spend_90d  spend_std_90d last_purchase_timestamp                 feedback_text region plan_type favorite_color browser_type phone_type  lucky_number  income_copy  constant_field  noise_01  noise_02  noise_03  noise_04  noise_05  noise_06  noise_07  noise_08  noise_09  noise_10  noise_11  noise_12  noise_13  noise_14  noise_15  noise_16  noise_17  noise_18  noise_19  noise_20  churn
     C00000   58    59758.463114  37619.726181             14           5            3      1128.267446     537.555086     2026-08-25 00:00:00   I want to cancel my account  North     Basic         Yellow      Firefox    Android            50 60356.047745               1  0.471897  0.540655 -0.718078 -0.834078 -1.040498 -0.559373 -0.674352  0.550352 -1.039817 -0.008503 -0.147872  0.197008 -1.504824 -0.402550 -0.850554 -0.327859  1.166833 -0.247191 -1.780402  1.590236      1
     C00001   5

## 3. Inspect missingness and class balance

In [15]:
print("Missing values:")
print(df.isna().sum().loc[lambda x: x > 0])
print("\nClass proportions:")
print(df.churn.value_counts(normalize=True).sort_index())
assert df.shape == (10000, 40)
assert df.customer_id.is_unique


Missing values:
monthly_income     104
monthly_debt        84
total_spend_90d     90
spend_std_90d      102
dtype: int64

Class proportions:
churn
0    0.3814
1    0.6186
Name: proportion, dtype: float64


## 4. Week 5 extraction, Week 6 construction and Week 7 selection
- Extract hour, weekday, weekend, night activity and recency from timestamps.
- Extract text length, complaint flag and a simple phrase-based sentiment proxy.
- Construct debt/income, return ratio, average order value and spending variability.
- Construct new-customer and high-debt flags, plus a debt × new-customer interaction.
- Retain nine domain-selected signals; exclude noise, duplicate income, constant fields and redundant representations.

The complaint rules only apply to the synthetic phrases here. Existing 90-day summaries are inputs, not event logs aggregated by this notebook. The 0.35 debt threshold is a fixed teaching assumption, never tuned using the test set.

In [16]:
class FeatureBuilder(BaseEstimator, TransformerMixin):
    """Domain rules fixed before evaluation; no target or test statistics used."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        z = X.copy()
        t = pd.to_datetime(z['last_purchase_timestamp'])
        z['purchase_hour'] = t.dt.hour
        z['purchase_day_of_week'] = t.dt.dayofweek
        z['is_weekend'] = (t.dt.dayofweek >= 5).astype(int)
        z['is_night'] = (t.dt.hour < 6).astype(int)
        z['recency_days'] = (pd.Timestamp('2026-11-01')-t).dt.total_seconds()/86400
        z['text_length'] = z['feedback_text'].fillna('').str.len()
        z['complaint_flag'] = z['feedback_text'].fillna('').str.contains('unhappy|poor|complaint|cancel', case=False, regex=True).astype(int)
        z['sentiment_score'] = np.where(z['complaint_flag'], -1, 1)
        for name, num, den in [('debt_to_income','monthly_debt','monthly_income'),
                               ('return_ratio','returns_90d','orders_90d'),
                               ('avg_order_value','total_spend_90d','orders_90d'),
                               ('spend_variability','spend_std_90d','total_spend_90d')]:
            z[name] = z[num]/z[den].replace(0, np.nan)
        z['new_customer'] = (z['tenure_months']<=6).astype(int)
        z['risk_interaction'] = z['debt_to_income']*z['new_customer']
        # Preserve missing ratios as missing rather than falsely labelling low risk.
        z['high_debt_burden'] = np.where(z['debt_to_income'].isna(), np.nan, (z['debt_to_income']>=.35).astype(float))
        return z.replace([np.inf,-np.inf],np.nan)

# Week 7: a priori domain selection. No correlations with held-out labels.
# Omit irrelevant noise/categories, ID, duplicate income, constant and raw text/date.
SELECTED = ['tenure_months','debt_to_income','return_ratio','spend_variability',
            'recency_days','is_night','complaint_flag','risk_interaction','high_debt_burden']



In [17]:
engineered_preview = FeatureBuilder().transform(df.drop(columns="churn"))
print(engineered_preview[SELECTED].head().to_string(index=False))
print("\nSelected final features:", SELECTED)

 tenure_months  debt_to_income  return_ratio  spend_variability  recency_days  is_night  complaint_flag  risk_interaction  high_debt_burden
            14        0.629530      0.600000           0.476443     68.000000         1               1               0.0               1.0
            15        0.214483      0.071429           0.808222     64.291667         0               0               0.0               0.0
            52        0.660256      0.153846                NaN     40.583333         0               0               0.0               1.0
            37        0.158164      0.075000           0.952361     47.625000         0               1               0.0               0.0
            60        0.300389      0.149254           0.292353     83.000000         1               1               0.0               0.0

Selected final features: ['tenure_months', 'debt_to_income', 'return_ratio', 'spend_variability', 'recency_days', 'is_night', 'complaint_flag', 'risk_interacti

## 5. Define the baseline and preprocessed pipelines
A literal zero-preprocessing logistic regression cannot consume text, dates and missing values. The **baseline** therefore uses minimum compatible conversion: epoch timestamps, whole-string one-hot encoding, imputation and scaling. It adds no semantic features, ratios, interactions or domain selection.

**Preprocessed model:** fixed row-wise feature construction, nine selected features, training-only median imputation and scaling, and the same logistic regression.

**Intermediate numeric ablation:** retains numeric originals and constructed features, including noise. It omits original categorical columns and raw text/date after extraction. Comparing it with the final model illustrates numeric feature selection; it is not a single-change ablation against the categorical baseline.

`fit_seconds` includes pipeline transformations and training. Prediction timing includes predict and predict_proba, so test transformations run twice. Overall runtime sums setup, fit and inference median components across three repetitions. Loading, dataset generation and metric computation are excluded.

In [20]:
def build_pipeline(X, mode):
    if mode == "baseline":
        # Minimum preprocessing required for logistic regression.
        work = X.drop(columns=["customer_id"]).copy()

        timestamps = pd.to_datetime(
            work["last_purchase_timestamp"],
            errors="coerce"
        )

        # Convert dates to Unix seconds while preserving missing dates.
        work["last_purchase_timestamp"] = (
            timestamps - pd.Timestamp("1970-01-01")
        ).dt.total_seconds()

        numeric_columns = work.select_dtypes(
            include="number"
        ).columns.tolist()

        categorical_columns = work.select_dtypes(
            exclude="number"
        ).columns.tolist()

        transform = ColumnTransformer([
            (
                "numeric",
                Pipeline([
                    ("impute", SimpleImputer(strategy="median")),
                    ("scale", StandardScaler())
                ]),
                numeric_columns
            ),
            (
                "category",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ),
                categorical_columns
            )
        ])

        pipeline = Pipeline([
            ("prepare", transform),
            (
                "model",
                LogisticRegression(
                    C=1,
                    max_iter=2000,
                    random_state=SEED
                )
            )
        ])

        return work, pipeline

    if mode not in ["engineered_all", "preprocessed"]:
        raise ValueError(
            "mode must be 'baseline', 'engineered_all', "
            "'preprocessed', or 'all' when calling run()."
        )

    if mode == "preprocessed":
        columns = SELECTED
    else:
        # Fixed row-wise feature extraction; no statistics are fitted here.
        engineered = FeatureBuilder().transform(X)
        columns = engineered.select_dtypes(
            include="number"
        ).columns.tolist()

    transform = ColumnTransformer([
        (
            "numeric",
            Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler())
            ]),
            columns
        )
    ])

    pipeline = Pipeline([
        ("engineer", FeatureBuilder()),
        ("prepare", transform),
        (
            "model",
            LogisticRegression(
                C=1,
                max_iter=2000,
                random_state=SEED
            )
        )
    ])

    return X, pipeline





In [21]:
def run(mode="all"):
    valid_modes = ["all", "baseline", "engineered_all", "preprocessed"]

    if mode not in valid_modes:
        raise ValueError(f"Choose one of: {valid_modes}")

    # Read the GitHub raw CSV URL stored in dataset.
    d = pd.read_csv(dataset)

    print("Dataset loaded successfully.")
    print("Shape:", d.shape)

    X = d.drop(columns="churn")
    y = d["churn"]

    train, test = train_test_split(
        np.arange(len(d)),
        test_size=0.25,
        random_state=SEED,
        stratify=y
    )

    modes = (
        ["baseline", "engineered_all", "preprocessed"]
        if mode == "all"
        else [mode]
    )

    rows = []

    for name in modes:
        timings = []

        for repeat in range(3):
            start = perf_counter()
            work, pipe = build_pipeline(X, name)
            setup_time = perf_counter() - start

            start = perf_counter()
            pipe.fit(work.iloc[train], y.iloc[train])
            fit_time = perf_counter() - start

            start = perf_counter()
            predictions = pipe.predict(work.iloc[test])
            probabilities = pipe.predict_proba(work.iloc[test])[:, 1]
            prediction_time = perf_counter() - start

            timings.append([setup_time, fit_time, prediction_time])

        median_times = np.median(timings, axis=0)

        rows.append({
            "experiment": name,
            "accuracy": accuracy_score(y.iloc[test], predictions),
            "f1": f1_score(y.iloc[test], predictions),
            "roc_auc": roc_auc_score(y.iloc[test], probabilities),
            "setup_seconds": median_times[0],
            "fit_seconds": median_times[1],
            "predict_and_probability_seconds": median_times[2],
            "total_seconds": median_times.sum(),
            "model_features": pipe.named_steps["model"].n_features_in_
        })

        print(
            f"\n{name}: "
            f"Accuracy = {rows[-1]['accuracy']:.2%}, "
            f"Runtime = {rows[-1]['total_seconds']:.4f} seconds"
        )
        print(
            "Confusion matrix:\n",
            confusion_matrix(y.iloc[test], predictions)
        )

    # These lines are inside the function, outside the loops.
    out = pd.DataFrame(rows)
    out.to_csv(f"results_{mode}.csv", index=False)

    return out

## 6. Run all experiments on exactly the same held-out customers
The stratified split contains 7,500 training rows and 2,500 test rows. All repetitions use the same split. We report median runtimes, not the best accuracy trial. Confusion matrix rows are actual labels and columns are predicted labels.

In [22]:
# dataset already contains your GitHub raw CSV URL.
comparison = run("all")
display(comparison)

Dataset loaded successfully.
Shape: (10000, 40)

baseline: Accuracy = 77.72%, Runtime = 0.1404 seconds
Confusion matrix:
 [[ 592  362]
 [ 195 1351]]

engineered_all: Accuracy = 83.20%, Runtime = 0.2671 seconds
Confusion matrix:
 [[ 702  252]
 [ 168 1378]]

preprocessed: Accuracy = 83.00%, Runtime = 0.1550 seconds
Confusion matrix:
 [[ 702  252]
 [ 173 1373]]


,experiment,accuracy,f1,roc_auc,setup_seconds,fit_seconds,predict_and_probability_seconds,total_seconds,model_features
0,baseline,0.7772,0.829089,0.843832,0.013708,0.092199,0.034478,0.140385,56
1,engineered_all,0.8320,0.867758,0.900168,0.068426,0.116804,0.081841,0.267071,46
2,preprocessed,0.8300,0.865973,0.900512,0.000036,0.082556,0.072375,0.154967,9


## 7. Interpret the measured improvement

In [23]:
# Run all experiments and store their results.
comparison = run(mode="all")

# Check that both required experiments are present.
required = {"baseline", "preprocessed"}

if not isinstance(comparison, pd.DataFrame):
    raise TypeError(
        "run() must return a DataFrame. "
        "Rerun the cell defining the updated run() function."
    )

if "experiment" not in comparison.columns:
    raise ValueError(
        "The results are missing the 'experiment' column."
    )

results = comparison.set_index("experiment")

missing = required - set(results.index)
if missing:
    raise ValueError(f"Missing experiment results: {sorted(missing)}")

baseline = results.loc["baseline"]
processed = results.loc["preprocessed"]

print(f"Baseline accuracy: {baseline['accuracy']:.2%}")
print(f"Preprocessed accuracy: {processed['accuracy']:.2%}")

gain = 100 * (processed["accuracy"] - baseline["accuracy"])
print(f"Accuracy gain: {gain:.2f} percentage points")

print(
    f"Baseline total runtime: "
    f"{baseline['total_seconds']:.4f} seconds"
)
print(
    f"Preprocessed total runtime: "
    f"{processed['total_seconds']:.4f} seconds"
)
print(
    f"Model inputs: {int(baseline['model_features'])}"
    f" -> {int(processed['model_features'])}"
)

print(
    "\nMore accurate does not necessarily mean faster. "
    "Removing features need not improve accuracy."
)

Dataset loaded successfully.
Shape: (10000, 40)

baseline: Accuracy = 77.72%, Runtime = 0.1434 seconds
Confusion matrix:
 [[ 592  362]
 [ 195 1351]]

engineered_all: Accuracy = 83.20%, Runtime = 0.2627 seconds
Confusion matrix:
 [[ 702  252]
 [ 168 1378]]

preprocessed: Accuracy = 83.00%, Runtime = 0.1569 seconds
Confusion matrix:
 [[ 702  252]
 [ 173 1373]]
Baseline accuracy: 77.72%
Preprocessed accuracy: 83.00%
Accuracy gain: 5.28 percentage points
Baseline total runtime: 0.1434 seconds
Preprocessed total runtime: 0.1569 seconds
Model inputs: 56 -> 9

More accurate does not necessarily mean faster. Removing features need not improve accuracy.


## 8. Classroom discussion
1. Why is debt-to-income more meaningful than absolute debt for this scenario?
2. Why does a linear model benefit from a threshold flag and interaction term?
3. Which extracted features did we drop, and why?
4. Does removing noise improve accuracy, runtime, or both in the intermediate comparison?
5. Why must imputation, scaling and category vocabularies be fitted on training data only?
6. Why would feature/threshold selection based on test results invalidate evaluation?

The earlier reference run measured 77.72% baseline and 83.00% final accuracy. The live outputs above are authoritative for your environment. Timing is illustrative and can fluctuate. To explore different features, use cross-validation on training rows and reserve a fresh test set for final evaluation.

